In [1]:
import pandas as pd

In [13]:
def read_file(file_path):
    if file_path.endswith('.csv'):
        return pd.read_csv(file_path)
    elif file_path.endswith('.xlsx'):
        return pd.read_excel(file_path)
    else:
        raise ValueError("Unsupported file format. Please provide a CSV or Excel file.")    
    return df

In [22]:
df = read_file("C:\\Users\\adham\\Downloads\\messy_ecommerce_sales_data.csv")

In [25]:
def mask_columns(df):
    original_columns = df.columns.tolist()
    masked_columns = [f"col_{i+1}" for i in range(len(original_columns))]
    mapping = dict(zip(masked_columns, original_columns))
    
    masked_df = df.copy()
    masked_df.columns = masked_columns
    return masked_df, mapping

In [26]:
masked_df, mapping = mask_columns(df)
print(masked_df.head())
print(mapping)

   col_1         col_2      col_3       col_4          col_5        col_6  \
0    100  Customer_100  ORD-41285  11/22/2024        Blender         Home   
1    101  Customer_101  ORD-35783    7/5/2025     Smartphone  Electronics   
2    102  Customer_102  ORD-84355  12/23/2024  Tennis Racket       Sports   
3    103  Customer_103  ORD-57811   3/19/2025        Science        Books   
4    104  Customer_104  ORD-93614  10/20/2025      Biography        Books   

  col_7   col_8             col_9      col_10   col_11  
0     3      38  Cash on Delivery     Shipped   114.00  
1     2     abd            PayPal  Processing      NaN  
2     1  389.05            PayPal   Delivered   389.05  
3     5  233.92            PayPal  Processing  1169.60  
4     1  552.51  Cash on Delivery  Processing   552.51  
{'col_1': 'ID', 'col_2': ' Customer_Name', 'col_3': 'Order_ID', 'col_4': 'Order_Date', 'col_5': 'Product', 'col_6': ' Category', 'col_7': 'Quantity', 'col_8': 'Price', 'col_9': 'Payment_Method', 

In [30]:
def detect_missing_values(df):
    missing_counts = df.isnull().sum()
    missing_percent = (missing_counts / len(df)) * 100
    result = pd.DataFrame({
        "missing_count": missing_counts,
        "missing_percent": missing_percent
    })
    return result[result["missing_count"] > 0] 

In [33]:
missing_report = detect_missing_values(masked_df)
print(missing_report)

        missing_count  missing_percent
col_6               8         7.766990
col_7               5         4.854369
col_8               5         4.854369
col_11             14        13.592233


In [34]:
def detect_duplicates(df):
    duplicate_count = df.duplicated().sum()
    duplicate_rows = df[df.duplicated()]
    return duplicate_count, duplicate_rows

In [39]:
dup_count, dup_rows = detect_duplicates(masked_df)
print(dup_count)
print(dup_rows)

1
     col_1         col_2      col_3     col_4       col_5       col_6 col_7  \
102    146  Customer_146  ORD-32755  7/9/2025  Basketball  electronic     2   

      col_8          col_9      col_10   col_11  
102  705.42  Bank Transfer  Processing  1410.84  


In [ ]:
def detect_type_issues(df):
    issues = {}
    for col in df.select_dtypes(include=["object", "str"]).columns:
        numeric_convertible = pd.to_numeric(df[col], errors="coerce")
        non_numeric_count = numeric_convertible.isnull().sum() - df[col].isnull().sum()
        non_null_count = df[col].notnull().sum()
        if 0 < non_numeric_count < (non_null_count * 0.5):
            issues[col] = non_numeric_count
    return issues

In [46]:
type_issues = detect_type_issues(masked_df)
print(type_issues)

{'col_7': np.int64(1), 'col_8': np.int64(4)}


In [47]:
def detect_outliers(df):
    outlier_report = {}
    for col in df.select_dtypes(include=["number"]).columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        if len(outliers) > 0:
            outlier_report[col] = len(outliers)
    return outlier_report
   

In [48]:
outlier_report = detect_outliers(masked_df)
print(outlier_report)

{'col_11': 3}


In [49]:
def detect_inconsistent_categories(df):
    inconsistency_report = {}
    for col in df.select_dtypes(include=["object", "str"]).columns:
        original_unique = df[col].dropna().unique()
        normalized_unique = df[col].dropna().str.strip().str.lower().unique()
        if len(normalized_unique) < len(original_unique):
            inconsistency_report[col] = len(original_unique) - len(normalized_unique)
    return inconsistency_report
     

In [50]:
inconsistency_report = detect_inconsistent_categories(masked_df)
print(inconsistency_report)

{'col_5': 1, 'col_6': 3}


In [62]:
def detect_whitespace_issues(df):
    whitespace_report = {}
    for col in df.select_dtypes(include=["object", "str"]).columns:
        has_whitespace = df[col].dropna().apply(lambda x: isinstance(x, str) and x != x.strip())
        count = has_whitespace.sum()
        if count > 0:
            whitespace_report[col] = int(count)
    return whitespace_report

In [63]:
whitespace_report = detect_whitespace_issues(masked_df)
print(whitespace_report)

{}


In [65]:
def detect_negative_values(df):
    negative_report = {}
    for col in df.select_dtypes(include=["number"]).columns:
        negative_count = (df[col] < 0).sum()
        if negative_count > 0:
            negative_report[col] = int(negative_count)
    return negative_report

In [66]:
negative_report = detect_negative_values(masked_df)
print(negative_report)

{'col_11': 3}


In [67]:
def detect_date_format_issues(df):
    import re
    date_format_report = {}
    for col in df.select_dtypes(include=["object", "str"]).columns:
        sample = df[col].dropna().astype(str)
        slash_format = sample.str.match(r'^\d{1,2}/\d{1,2}/\d{4}$').sum()
        dash_format = sample.str.match(r'^\d{4}-\d{1,2}-\d{1,2}$').sum()
        if slash_format > 0 and dash_format > 0:
            date_format_report[col] = {"slash_format": int(slash_format), "dash_format": int(dash_format)}
    return date_format_report

In [68]:
data_format = detect_date_format_issues(masked_df)
print(data_format)

{}


In [60]:
def generate_report(df):
    report = {
        "missing_values": detect_missing_values(df),
        "duplicates": detect_duplicates(df)[0],
        "type_issues": detect_type_issues(df),
        "outliers": detect_outliers(df),
        "inconsistent_categories": detect_inconsistent_categories(df)
    }
    return report

In [61]:
full_report = generate_report(masked_df)
print(full_report)

{'missing_values':         missing_count  missing_percent
col_6               8         7.766990
col_7               5         4.854369
col_8               5         4.854369
col_11             14        13.592233, 'duplicates': np.int64(1), 'type_issues': {'col_7': np.int64(1), 'col_8': np.int64(4)}, 'outliers': {'col_11': 3}, 'inconsistent_categories': {'col_5': 1, 'col_6': 3}}
